# Google Play Store App Scraper

This notebook demonstrates how to scrape app data from the Google Play Store using the `google_play_scraper` package. We'll create functions to search for apps by category, extract app information, and analyze the data.

## Features:
- Search apps by category with optimized queries
- Extract comprehensive app information
- Display results in a formatted way
- Analyze and visualize app data
- Export data to CSV for further analysis

## 1. Install Required Libraries

First, we need to install the necessary packages. Run the cell below to install the required dependencies.

In [ ]:
# Install required packages
!pip install google-play-scraper pandas matplotlib seaborn

## 2. Import Libraries and Setup

Import all necessary libraries for scraping, data analysis, and visualization.

In [ ]:
# Import required libraries
from google_play_scraper import search, app
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('default')
sns.set_palette("husl")

print("All libraries imported successfully!")

## 3. Define Category Search Function

Create a comprehensive function to search for apps by category with optimized search queries.

In [ ]:
def list_apps_by_category(category='GAME', num_apps=10):
    """
    Create a list of apps from a given category
    
    Args:
        category (str): The category to search for apps (e.g., 'GAME', 'BUSINESS', 'EDUCATION', 'SOCIAL', 'ENTERTAINMENT')
        num_apps (int): Number of apps to retrieve (default: 10)
    
    Returns:
        list: List of app dictionaries containing app information
    """
    
    # Category-specific search queries to improve results
    category_queries = {
        'GAME': 'game',
        'BUSINESS': 'business productivity',
        'EDUCATION': 'education learning',
        'SOCIAL': 'social media',
        'ENTERTAINMENT': 'entertainment video',
        'HEALTH_AND_FITNESS': 'health fitness',
        'LIFESTYLE': 'lifestyle',
        'MUSIC_AND_AUDIO': 'music audio',
        'NEWS_AND_MAGAZINES': 'news magazine',
        'PHOTOGRAPHY': 'photo camera',
        'SHOPPING': 'shopping',
        'SPORTS': 'sports',
        'TOOLS': 'tools utility',
        'TRAVEL_AND_LOCAL': 'travel maps',
        'WEATHER': 'weather'
    }
    
    try:
        # Use category-specific query if available, otherwise use the category name
        query = category_queries.get(category.upper(), category.lower())
        
        # Search for apps in the specified category
        apps = search(
            query=query,
            lang="en",  # Language
            country="us",  # Country
            n_hits=num_apps  # Number of results
        )
        
        app_list = []
        
        for app in apps:
            app_info = {
                'title': app.get('title', 'N/A'),
                'appId': app.get('appId', 'N/A'),
                'developer': app.get('developer', 'N/A'),
                'rating': app.get('score', 'N/A'),
                'installs': app.get('installs', 'N/A'),
                'free': app.get('free', 'N/A'),
                'category': app.get('genre', 'N/A')
            }
            app_list.append(app_info)
        
        return app_list
        
    except Exception as e:
        print(f"Error fetching apps: {e}")
        return []

print("Category search function defined successfully!")

## 4. Create App Display Function

Create a function to display the scraped app data in a formatted, readable way.

In [ ]:
def print_apps(apps):
    """
    Print the list of apps in a formatted way
    
    Args:
        apps (list): List of app dictionaries
    """
    if not apps:
        print("No apps found.")
        return
    
    print(f"\nFound {len(apps)} apps:")
    print("=" * 80)
    
    for i, app in enumerate(apps, 1):
        print(f"\n{i}. {app['title']}")
        print(f"   📱 Package: {app['appId']}")
        print(f"   👩‍💻 Developer: {app['developer']}")
        print(f"   ⭐ Rating: {app['rating']}")
        print(f"   📥 Installs: {app['installs']}")
        print(f"   💰 Free: {'Yes' if app['free'] else 'No'}")
        print(f"   📂 Category: {app['category']}")
        print("-" * 80)

def create_dataframe(apps):
    """
    Convert app list to pandas DataFrame for analysis
    
    Args:
        apps (list): List of app dictionaries
    
    Returns:
        pd.DataFrame: DataFrame containing app data
    """
    if not apps:
        return pd.DataFrame()
    
    df = pd.DataFrame(apps)
    return df

print("Display and DataFrame functions defined successfully!")

## 5. Scrape Apps by Category

Now let's test our functions by scraping apps from different categories.

In [ ]:
# Example 1: Scrape Game apps
print("🎮 Scraping GAME apps...")
game_apps = list_apps_by_category(category='GAME', num_apps=5)
print_apps(game_apps)

In [ ]:
# Example 2: Scrape Business apps
print("💼 Scraping BUSINESS apps...")
business_apps = list_apps_by_category(category='BUSINESS', num_apps=5)
print_apps(business_apps)

## 6. Data Analysis and Visualization

Let's analyze the scraped data and create visualizations to gain insights.

In [ ]:
# Collect data from multiple categories for analysis
categories = ['GAME', 'BUSINESS', 'EDUCATION', 'SOCIAL', 'ENTERTAINMENT']
all_apps = []

print("Collecting apps from multiple categories...")
for category in categories:
    print(f"Scraping {category} apps...")
    apps = list_apps_by_category(category=category, num_apps=10)
    all_apps.extend(apps)

# Create DataFrame for analysis
df = create_dataframe(all_apps)
print(f"\nCollected {len(df)} apps total")
print("\nDataFrame Info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Google Play Store Apps Analysis', fontsize=16)

# 1. Rating distribution
axes[0, 0].hist(df['rating'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('App Rating Distribution')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Number of Apps')

# 2. Category distribution
category_counts = df['category'].value_counts()
axes[0, 1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%')
axes[0, 1].set_title('Apps by Category')

# 3. Free vs Paid apps
free_counts = df['free'].value_counts()
axes[1, 0].bar(['Free', 'Paid'], [free_counts.get(True, 0), free_counts.get(False, 0)])
axes[1, 0].set_title('Free vs Paid Apps')
axes[1, 0].set_ylabel('Number of Apps')

# 4. Top developers by number of apps
top_developers = df['developer'].value_counts().head(10)
axes[1, 1].barh(range(len(top_developers)), top_developers.values)
axes[1, 1].set_yticks(range(len(top_developers)))
axes[1, 1].set_yticklabels(top_developers.index)
axes[1, 1].set_title('Top 10 Developers by App Count')
axes[1, 1].set_xlabel('Number of Apps')

plt.tight_layout()
plt.show()

## 7. Export Data to CSV

Save the scraped app data to a CSV file for further analysis or reporting.

In [ ]:
# Export data to CSV
csv_filename = 'google_play_apps_data.csv'
df.to_csv(csv_filename, index=False)
print(f"✅ Data exported to {csv_filename}")

# Display summary statistics
print("\n📊 Summary Statistics:")
print("=" * 50)
print(f"Total apps scraped: {len(df)}")
print(f"Average rating: {df['rating'].mean():.2f}")
print(f"Free apps: {df['free'].sum()}")
print(f"Paid apps: {len(df) - df['free'].sum()}")
print(f"Unique categories: {df['category'].nunique()}")
print(f"Unique developers: {df['developer'].nunique()}")

print("\n🏆 Top 5 Highest Rated Apps:")
top_rated = df.nlargest(5, 'rating')[['title', 'developer', 'rating', 'category']]
for i, (_, app) in enumerate(top_rated.iterrows(), 1):
    print(f"{i}. {app['title']} by {app['developer']} - ⭐{app['rating']} ({app['category']})")

## Conclusion

This notebook demonstrates a complete Google Play Store scraping workflow:

1. **Data Collection**: Successfully scraped app information from multiple categories
2. **Data Processing**: Structured the data into a pandas DataFrame for analysis
3. **Visualization**: Created insightful charts showing app distributions and trends
4. **Export**: Saved the data to CSV for further use

### Key Features of Our Scraper:
- ✅ Category-specific search optimization
- ✅ Comprehensive app information extraction
- ✅ Error handling and data validation
- ✅ Data analysis and visualization
- ✅ Export functionality

### Next Steps:
- Expand to more categories
- Add more detailed app information (descriptions, screenshots, etc.)
- Implement sentiment analysis on app reviews
- Create automated reporting dashboards